In [0]:
import requests
from pathlib import Path
import json
import time
import os

BASE_URL = "https://api.spela.svenskaspel.se/draw/1/stryktipset/draws"
BRONZE_DIR = Path("/Volumes/stryktipset/bronze/raw_draws")

In [0]:
def get_latest_draw_number() -> int:
    i = 0
    while i < 3:
        try:
            response = requests.get(BASE_URL, timeout=30)
            if response.ok:
                draws = response.json()['draws']
                max_draw = max(d['drawNumber'] for d in draws)
                print(f"Latest draw number: {max_draw}")
                return max_draw

            else:
                raise RuntimeError(f"Failed to fetch latest draw number: {response.status_code}")
            
        except requests.exceptions.RequestException as e:
            i += 1
            if i == 3:
                raise RuntimeError(f"Failed to fetch latest draw number after 3 attempts: {e}") from e
            time.sleep(0.5)

In [0]:
def draw_already_saved(draw_number) -> bool:
    file_path = BRONZE_DIR / f'draw_{draw_number}.json' 
    if file_path.exists():
        return True
    else:
        return False

In [0]:
def fetch_draw(draw_number) -> dict | None:
    i = 0
    while i < 3:
        try:
            response = requests.get(f'{BASE_URL}/{draw_number}', timeout=30)
            if response.ok:
                draw = response.json()
                return draw
                
            else:
                print(f"Failed draw {draw_number}: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            i += 1
            
            if i == 3:
                print(f"Failed to fetch {draw_number} after 3 attempts: {e}")
                return None

            time.sleep(0.5)

In [0]:
def save_raw(draw_number, data) -> None:
    out_path = BRONZE_DIR / f"draw_{draw_number}.json"
    tmp_path = out_path.with_suffix(".json.tmp")

    tmp_path.write_text(json.dumps(data, indent=2), encoding="utf-8")
    os.replace(tmp_path, out_path)

In [0]:
def main():
    fetched = 0
    skipped = 0
    failed = []
    for draw_number in range(4267, get_latest_draw_number()):
        if draw_already_saved(draw_number):
            skipped += 1
            continue
        draw = fetch_draw(draw_number)
        if draw is not None:
            fetched += 1
            save_raw(draw_number, draw)
        else:
            failed.append(draw_number)

    print(f"Number of fetched draws: {fetched}")
    print(f"Number of skipped draws: {skipped}")
    print(f"Failed draws: {failed}")

    if failed:
        raise RuntimeError(f"Bronze ingest failed for {len(failed)} draw(s): {failed}")
            
main()